# bronze__ingerir__balancos_internacionais_argentina

Camada: Bronze  
Fonte: Secretaría de Energía da República Argentina (API CKAN)  


Objetivo:
- Ingestão dos dados de produção das bacias sedimentares (Austral, Golfo San Jorge,Neuquina, Noroeste e Cuyana) a partir de 2019 até o período corrente, garantindo atualização contínua sempre que novos meses/registros forem disponibilizados na origem, adicionando metadados técnicos de ingestão para rastreabilidade no Lakehouse.

## Metadados adicionados

- `dt_ingestao`
- `fonte`
- `canal_ingestao`
- `url_origem`

## Entradas

- API Datos Energía Argentina (Recurso: 876b3746-85e2-4039-adeb-b1354436159f)

## Saídas

- `datalake_bronze_dgn_v01.balancos_internacionais.producao_petroleo_gas_bacias_argentina` (Delta)
- `Files/Arquivos_brutos/producao_petroleo_gas_bacias_argentina/producao_petroleo_gas_bacias_argentina.csv`
---

## Importação de bibliotecas

In [1]:
import requests
import pandas as pd
from IPython.display import display
import datetime
import numpy as np

StatementMeta(, a6718d36-97b6-4dc4-96de-9f65ab1004c3, 3, Finished, Available, Finished, False)

## Lista de Bacias Monitoradas

In [2]:
bacias_alvo = [
    "AUSTRAL",
    "GOLFO SAN JORGE",
    "NEUQUINA",
    "NOROESTE",
    "CUYANA"
]

df_lista_bacias = pd.DataFrame({"bacia": bacias_alvo})

print(f"Total de bacias a monitorar: {len(df_lista_bacias)}")
display(df_lista_bacias)

StatementMeta(, a6718d36-97b6-4dc4-96de-9f65ab1004c3, 4, Finished, Available, Finished, False)

Total de bacias a monitorar: 5


,bacia
0,AUSTRAL
1,GOLFO SAN JORGE
2,NEUQUINA
3,NOROESTE
4,CUYANA


## Coleta da Produção por Bacia via API CKAN SQL

In [3]:
def obter_producao_gas_bacia(
    bacia: str,
    ano_inicial: int = 2019,
    ano_final: int = None,
    resource_id: str = "876b3746-85e2-4039-adeb-b1354436159f"
) -> pd.DataFrame:
    """
    Coleta dados mensais agregados unicamente de gás natural via API CKAN SQL
    a partir de 2019 até ao ano corrente.
    """
    base_url = "http://datos.energia.gob.ar/api/3/action/datastore_search_sql"

    if ano_final is None:
        ano_final = datetime.date.today().year

    # Consulta filtrada apenas para volume de gás natural
    sql_query = f"""
    SELECT 
        cuenca,
        CAST(anio AS INTEGER) AS anio,
        CAST(mes AS INTEGER) AS mes,
        SUM(CAST(prod_gas AS NUMERIC)) AS total_gas_miles_m3
    FROM "{resource_id}"
    WHERE UPPER(cuenca) = '{bacia.upper()}'
      AND CAST(anio AS INTEGER) >= {ano_inicial}
      AND CAST(anio AS INTEGER) <= {ano_final}
    GROUP BY cuenca, anio, mes
    ORDER BY anio ASC, mes ASC
    """

    params = {"sql": sql_query}

    response = requests.get(base_url, params=params, timeout=120)
    response.raise_for_status()

    payload = response.json()
    if not payload.get("success"):
        raise ValueError(f"Erro na API da Secretaría de Energía: {payload.get('error')}")

    records = payload.get("result", {}).get("records", [])
    return pd.DataFrame(records)


ano_inicial = 2019
ano_final = datetime.date.today().year

StatementMeta(, a6718d36-97b6-4dc4-96de-9f65ab1004c3, 5, Finished, Available, Finished, False)

## Período dinâmico iniciando em 2019

In [4]:
ano_inicial = 2019
ano_final = datetime.date.today().year

StatementMeta(, a6718d36-97b6-4dc4-96de-9f65ab1004c3, 6, Finished, Available, Finished, False)

## Teste unitário de chamada

In [5]:
df_teste_bacia = obter_producao_gas_bacia("NEUQUINA", ano_inicial=ano_inicial, ano_final=ano_final)
display(df_teste_bacia.head())

StatementMeta(, a6718d36-97b6-4dc4-96de-9f65ab1004c3, 7, Finished, Available, Finished, False)

,anio,cuenca,total_gas_miles_m3,mes
0,2022,NEUQUINA,2702505.946009894,1
1,2022,NEUQUINA,2353589.253422061,2
2,2022,NEUQUINA,2709280.889337422000288,3
3,2022,NEUQUINA,2536333.451006802,4
4,2022,NEUQUINA,2901156.541559053,5


## Ingestão Consolidada com Metadados Técnicos

In [6]:
dfs = []

dt_ingestao = pd.Timestamp.now(datetime.timezone.utc)
fonte = "secretaria_energia_argentina"
canal_ingestao = "api_ckan_datastore_search_sql"
url_origem = "http://datos.energia.gob.ar/api/3/action/datastore_search_sql"

for bacia in df_lista_bacias["bacia"].unique():
    print(f"A recolher dados de gás da bacia: {bacia} ({ano_inicial} a {ano_final})...")

    try:
        df_bacia = obter_producao_gas_bacia(
            bacia=bacia,
            ano_inicial=ano_inicial,
            ano_final=ano_final
        )

        if df_bacia.empty:
            print(f"Nenhum dado retornado para a bacia {bacia}")
            continue

        colunas_numericas = ["anio", "mes", "total_gas_miles_m3"]
        for col in colunas_numericas:
            if col in df_bacia.columns:
                df_bacia[col] = pd.to_numeric(df_bacia[col], errors="coerce")

        df_bacia["dt_ingestao"] = dt_ingestao
        df_bacia["fonte"] = fonte
        df_bacia["canal_ingestao"] = canal_ingestao
        df_bacia["url_origem"] = url_origem

        dfs.append(df_bacia)

    except Exception as e:
        print(f"Erro ao recolher dados da bacia {bacia}: {e}")

df_gas_bacias_bronze = pd.concat(dfs, ignore_index=True)

StatementMeta(, a6718d36-97b6-4dc4-96de-9f65ab1004c3, 8, Finished, Available, Finished, False)

A recolher dados de gás da bacia: AUSTRAL (2019 a 2026)...
A recolher dados de gás da bacia: GOLFO SAN JORGE (2019 a 2026)...
A recolher dados de gás da bacia: NEUQUINA (2019 a 2026)...
A recolher dados de gás da bacia: NOROESTE (2019 a 2026)...
A recolher dados de gás da bacia: CUYANA (2019 a 2026)...


In [7]:
df_gas_bacias_bronze.info()
display(df_gas_bacias_bronze.tail(15))

StatementMeta(, a6718d36-97b6-4dc4-96de-9f65ab1004c3, 9, Finished, Available, Finished, False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   anio                60 non-null     int64              
 1   cuenca              60 non-null     object             
 2   total_gas_miles_m3  60 non-null     float64            
 3   mes                 60 non-null     int64              
 4   dt_ingestao         60 non-null     datetime64[us, UTC]
 5   fonte               60 non-null     object             
 6   canal_ingestao      60 non-null     object             
 7   url_origem          60 non-null     object             
dtypes: datetime64[us, UTC](1), float64(1), int64(2), object(4)
memory usage: 3.9+ KB


,anio,cuenca,total_gas_miles_m3,mes,dt_ingestao,fonte,canal_ingestao,url_origem
45,2022,NOROESTE,117547.169701,10,2026-09-29 13:23:06.836882+00:00,secretaria_energia_argentina,api_ckan_datastore_search_sql,http://datos.energia.gob.ar/api/3/action/datas...
46,2022,NOROESTE,112334.852912,11,2026-09-29 13:23:06.836882+00:00,secretaria_energia_argentina,api_ckan_datastore_search_sql,http://datos.energia.gob.ar/api/3/action/datas...
47,2022,NOROESTE,115288.323306,12,2026-09-29 13:23:06.836882+00:00,secretaria_energia_argentina,api_ckan_datastore_search_sql,http://datos.energia.gob.ar/api/3/action/datas...
48,2022,CUYANA,4341.963315,1,2026-09-29 13:23:06.836882+00:00,secretaria_energia_argentina,api_ckan_datastore_search_sql,http://datos.energia.gob.ar/api/3/action/datas...
49,2022,CUYANA,3844.682707,2,2026-09-29 13:23:06.836882+00:00,secretaria_energia_argentina,api_ckan_datastore_search_sql,http://datos.energia.gob.ar/api/3/action/datas...
50,2022,CUYANA,4376.325562,3,2026-09-29 13:23:06.836882+00:00,secretaria_energia_argentina,api_ckan_datastore_search_sql,http://datos.energia.gob.ar/api/3/action/datas...
51,2022,CUYANA,4122.619872,4,2026-09-29 13:23:06.836882+00:00,secretaria_energia_argentina,api_ckan_datastore_search_sql,http://datos.energia.gob.ar/api/3/action/datas...
52,2022,CUYANA,4385.337604,5,2026-09-29 13:23:06.836882+00:00,secretaria_energia_argentina,api_ckan_datastore_search_sql,http://datos.energia.gob.ar/api/3/action/datas...
53,2022,CUYANA,4099.621218,6,2026-09-29 13:23:06.836882+00:00,secretaria_energia_argentina,api_ckan_datastore_search_sql,http://datos.energia.gob.ar/api/3/action/datas...
54,2022,CUYANA,4122.961995,7,2026-09-29 13:23:06.836882+00:00,secretaria_energia_argentina,api_ckan_datastore_search_sql,http://datos.energia.gob.ar/api/3/action/datas...


## Gravação no Medalhão Bronze (Delta Lake & CSV Backup)

### Conversão do DataFrame pandas para Spark DataFrame do Fabric Lakehouse

In [8]:
df_spark = spark.createDataFrame(df_gas_bacias_bronze)

NOME_TABELA = "datalake_bronze_dgn_v01.balancos_internacionais.producao_gas_natural_bacias_argentina"
CAMINHO_DIRETORIO_CSV = "Files/Arquivos_brutos/argentina/producao_gas_natural_bacias_argentina"
CAMINHO_TEMP = f"{CAMINHO_DIRETORIO_CSV}/temp_csv"
CAMINHO_ARQUIVO_FINAL = f"{CAMINHO_DIRETORIO_CSV}/producao_gas_natural_bacias_argentina.csv"

try:
    print("Gravando tabela formato Delta.")
    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(NOME_TABELA)
    print(f"Tabela {NOME_TABELA} atualizada com sucesso.")

    print("Exportando backup em formato CSV.")
    df_spark.coalesce(1).write \
        .format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .option("delimiter", ";") \
        .save(CAMINHO_TEMP)

    arquivos_temp = mssparkutils.fs.ls(CAMINHO_TEMP)
    arquivo_part = None
    for arquivo in arquivos_temp:
        if arquivo.name.endswith(".csv"):
            arquivo_part = arquivo.path
            break

    if arquivo_part:
        mssparkutils.fs.cp(arquivo_part, CAMINHO_ARQUIVO_FINAL, True)
        mssparkutils.fs.rm(CAMINHO_TEMP, True)
        print(f"Arquivo CSV salvo com sucesso em: {CAMINHO_ARQUIVO_FINAL}")
    else:
        print("Aviso: O arquivo CSV não foi detectado na estrutura temporária.")

    print("\nProcessamento do Medalhão Bronze concluído com sucesso.")

except Exception as e:
    print(f"\nFALHA CRÍTICA DE I/O: Erro ao gravar dados no Fabric Lakehouse: {e}")
    raise e

StatementMeta(, a6718d36-97b6-4dc4-96de-9f65ab1004c3, 10, Finished, Available, Finished, False)

Gravando tabela formato Delta.
Tabela datalake_bronze_dgn_v01.balancos_internacionais.producao_gas_natural_bacias_argentina atualizada com sucesso.
Exportando backup em formato CSV.
Arquivo CSV salvo com sucesso em: Files/Arquivos_brutos/argentina/producao_gas_natural_bacias_argentina/producao_gas_natural_bacias_argentina.csv

Processamento do Medalhão Bronze concluído com sucesso.
